# 🎤 3 · Add your voice to your tracks

Takes your trained voice model (notebook 2) and **replaces the vocals on your songs with your voice**.

**Replacing the vocals on finished songs = `full_song` mode (the default).** Put the songs in `MyVoice/tracks`, set your model name and run.
Recommended routine: keep the 20 s preview on, adjust `PITCH_SHIFT` until it sounds right, then set `PREVIEW_SECONDS = 0` for the full song.

The notebook supports two modes:

| Mode | You provide | What happens |
|---|---|---|
| `full_song` | Finished songs with a vocal in `MyVoice/tracks` | Demucs splits vocal / instrumental → vocal becomes your voice → mixed back over the original instrumental |
| `vocal_plus_instrumental` | `MyVoice/vocals/<song>.wav` + `MyVoice/instrumentals/<song>.wav` | Your guide vocal (or acapella) is converted to your voice (optional) and mixed over the instrumental |

For each song you get, in `MyVoice/output/<song>/`: the final mix, your vocal alone, and the instrumental alone, so you can re-mix in your DAW.

**Runtime → Change runtime type → T4 GPU** before you start.

> Only use tracks you have the rights to, and only voice models of voices you have permission to use (e.g. your own).

In [ ]:
#@title 📁 Connect Google Drive and create folders
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

VOICE_ROOT = Path("/content/drive/MyDrive/MyVoice")   # everything lives here, so it survives Colab resets
FOLDERS = {name: VOICE_ROOT / name for name in
           ["recordings", "dataset", "models", "checkpoints", "tracks", "instrumentals", "vocals", "output"]}
for p in FOLDERS.values():
    p.mkdir(parents=True, exist_ok=True)
print("Your folders:")
for name, p in FOLDERS.items():
    print(f"  {name:14s} {p}")

In [ ]:
import os, re, shutil, subprocess, sys, time
from pathlib import Path

def run(cmd, cwd=None, env=None):
    """Run a command, stream its output live, and raise if it fails."""
    cmd = [str(c) for c in cmd]
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")

if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 🔧 Install the voice engine (Applio / RVC) - about 5 min the first time
APPLIO_REPO = "https://github.com/IAHispano/Applio"
APPLIO_REF = "c7665ac9a305b3683570ed914f1577d53d4b75c4"   #@param {type:"string"}
# ^ a commit these notebooks were written against. Type "main" to try the newest Applio instead.
APPLIO_DIR = Path("/content/Applio")

if not APPLIO_DIR.exists():
    run(["git", "clone", "-q", APPLIO_REPO, APPLIO_DIR])
run(["git", "checkout", "-q", APPLIO_REF], cwd=APPLIO_DIR)

marker = APPLIO_DIR / ".installed"
if marker.exists() and marker.read_text() == APPLIO_REF:
    print("Applio already installed in this session.")
else:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "portaudio19-dev"])
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    run([sys.executable, "-m", "uv", "pip", "install", "--system", "-q", "-r", "requirements.txt",
         "--extra-index-url", "https://download.pytorch.org/whl/cu128",
         "--index-strategy", "unsafe-best-match"], cwd=APPLIO_DIR)
    run([sys.executable, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--exe"], cwd=APPLIO_DIR)
    shutil.copy(APPLIO_DIR / "assets" / "config_template.json", APPLIO_DIR / "assets" / "config.json")
    marker.write_text(APPLIO_REF)
    print("Applio installed.")

In [ ]:
#@title 🔧 Install the vocal separator (Demucs) - only needed for "full_song" mode
# Demucs goes in its own isolated Python environment so it can't clash with Applio's packages.
DEMUCS_ENV = Path("/content/demucs_env")
DEMUCS_PY = DEMUCS_ENV / "bin" / "python"
if not DEMUCS_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", DEMUCS_ENV])
    run([sys.executable, "-m", "uv", "pip", "install", "-q", "--python", DEMUCS_PY,
         "torch==2.5.1", "torchaudio==2.5.1", "demucs==4.0.1", "soundfile", "numpy<2"])
print("Demucs ready.")

## Settings
Change these and re-run from here as often as you like - separation results are cached, so tweaking is quick.

In [ ]:
#@title ⚙️ Settings
MODEL_NAME = "myvoice"          #@param {type:"string"}
MODE = "full_song"              #@param ["full_song", "vocal_plus_instrumental"]
# full_song               : songs in MyVoice/tracks that already contain a vocal (any singer).
#                           The vocal is split out, turned into YOUR voice, and mixed back over the original instrumental.
# vocal_plus_instrumental : separate files with matching names, e.g. MyVoice/vocals/song1.wav (a guide vocal / acapella)
#                           and MyVoice/instrumentals/song1.wav. The vocal is converted (optional) and mixed over the instrumental.

CONVERT_VOCALS = True           #@param {type:"boolean"}
# Untick in vocal_plus_instrumental mode if the vocal is already you (you sang it yourself) and you only want the mix/polish.

#@markdown ### Voice conversion
PITCH_SHIFT = 0                 #@param {type:"slider", min:-12, max:12, step:1}
#@markdown Semitones. 0 if the source singer has a similar range to you; about +12 if the source is male and you are female; about -12 for the opposite. Fine-tune by ear.
INDEX_RATE = 0.5                #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Higher = sounds more like your recordings, but can add artifacts/accent leakage.
PROTECT = 0.33                  #@param {type:"slider", min:0.0, max:0.5, step:0.01}
#@markdown Protects consonants and breaths. Lower = more protection (0.5 = off).
F0_METHOD = "rmvpe"             #@param ["rmvpe", "fcpe", "crepe", "hybrid[rmvpe+fcpe]"]
SPLIT_AUDIO = True              #@param {type:"boolean"}
#@markdown Converts long vocals in chunks - recommended for full songs.
SEPARATION_MODEL = "htdemucs"   #@param ["htdemucs", "htdemucs_ft"]
#@markdown htdemucs_ft is slightly cleaner but about 4x slower.

#@markdown ### Preview (recommended while tuning)
PREVIEW_SECONDS = 20            #@param {type:"slider", min:0, max:60, step:5}
PREVIEW_START = 30              #@param {type:"integer"}
#@markdown Only converts and mixes this many seconds, starting at PREVIEW_START (seconds into the song) - pick a part with singing.
#@markdown Set PREVIEW_SECONDS to **0** for the full song once it sounds right.

#@markdown ### Mix
VOCAL_DB = 0.0                  #@param {type:"slider", min:-12, max:12, step:0.5}
INSTRUMENTAL_DB = 0.0           #@param {type:"slider", min:-12, max:12, step:0.5}
VOCAL_OFFSET_MS = 0             #@param {type:"slider", min:-300, max:300, step:5}
#@markdown Positive = vocal later, negative = earlier. Mostly useful for vocals you recorded yourself.
REVERB = 0.12                   #@param {type:"slider", min:0.0, max:0.5, step:0.01}
COMPRESS_VOCAL = True           #@param {type:"boolean"}
TARGET_LUFS = -14.0             #@param {type:"slider", min:-24, max:-6, step:0.5}
OUTPUT_FORMAT = "wav"           #@param ["wav", "mp3"]

MODEL_DIR = FOLDERS["models"] / MODEL_NAME
PTH, INDEX = MODEL_DIR / f"{MODEL_NAME}.pth", MODEL_DIR / f"{MODEL_NAME}.index"
for f in (PTH, INDEX):
    if not f.exists():
        raise SystemExit(f"Missing {f}. Finish notebook 02 (export step) or fix MODEL_NAME.")
print("Using model:", PTH)

In [ ]:
#@title 🧰 Mixing helpers (just run this)
try:
    import pyloudnorm
except ImportError:
    run([sys.executable, "-m", "pip", "install", "-q", "pyloudnorm"])
import numpy as np
import soundfile as sf
import soxr
from pedalboard import Compressor, HighpassFilter, Limiter, Pedalboard, Reverb

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".webm", ".mp4", ".mov", ".wma", ".aiff", ".aif"}
SR = 44100
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)

def list_audio(folder):
    return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXTS)

def to_wav(src, dst, sr=SR, channels=2):
    """Any audio file -> float32 WAV (needs ffmpeg, which Colab has)."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-vn", "-ac", str(channels),
                    "-ar", str(sr), "-c:a", "pcm_f32le", str(dst)], check=True)

def load_stereo(path, sr=SR):
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    if file_sr != sr:
        x = soxr.resample(x, file_sr, sr)
    if x.shape[1] == 1:
        x = np.repeat(x, 2, axis=1)
    return np.ascontiguousarray(x[:, :2])

def fit_length(x, n):
    return x[:n] if len(x) >= n else np.pad(x, ((0, n - len(x)), (0, 0)))

def apply_board(x, board, sr=SR):
    """x is (samples, 2); pedalboard wants (channels, samples)."""
    return np.ascontiguousarray(board(np.ascontiguousarray(x.T), sr).T)

def mix_vocal_over_instrumental(vocal_path, inst_path, vocal_db=0.0, inst_db=0.0, offset_ms=0.0,
                                reverb=0.12, compress=True, target_lufs=-14.0):
    """Polish the vocal, put it over the instrumental, normalise loudness. Returns (mix, vocal, instrumental)."""
    v, inst = load_stereo(vocal_path), load_stereo(inst_path)
    off = int(round(offset_ms / 1000 * SR))
    if off > 0:
        v = np.pad(v, ((off, 0), (0, 0)))
    elif off < 0:
        v = v[-off:]
    v = fit_length(v, len(inst))

    chain = [HighpassFilter(cutoff_frequency_hz=80)]
    if compress:
        chain.append(Compressor(threshold_db=-20, ratio=3, attack_ms=10, release_ms=120))
    if reverb > 0:
        chain.append(Reverb(room_size=0.3, damping=0.6, wet_level=reverb, dry_level=1.0, width=1.0))
    v = apply_board(v, Pedalboard(chain))
    v = v * 10 ** (vocal_db / 20)
    inst = inst * 10 ** (inst_db / 20)
    mix = v + inst

    if target_lufs is not None:
        try:
            import pyloudnorm as pyln
            loud = pyln.Meter(SR).integrated_loudness(mix)
            if np.isfinite(loud):
                mix = mix * 10 ** ((target_lufs - loud) / 20)
        except Exception as e:                       # e.g. clip shorter than 0.4 s
            print("  (loudness normalisation skipped:", e, ")")
    mix = apply_board(mix, Pedalboard([Limiter(threshold_db=-1.0, release_ms=100)]))
    return mix, v, inst

def save_audio(x, path, fmt):
    """Save as 24-bit WAV, or 320k MP3."""
    path = Path(path)
    wav = path.with_suffix(".wav")
    sf.write(str(wav), x, SR, subtype="PCM_24")
    if fmt == "mp3":
        mp3 = path.with_suffix(".mp3")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav), "-c:a", "libmp3lame",
                        "-b:a", "320k", str(mp3)], check=True)
        wav.unlink()
        return mp3
    return wav

## Go

In [ ]:
#@title ▶️ Process the songs
from IPython.display import Audio, display

SEP_DIR = Path("/content/work/separated")

def separate(track):
    """Return (vocals.wav, instrumental.wav) for a full song using Demucs."""
    out = SEP_DIR / SEPARATION_MODEL / track.stem
    vocals, inst = out / "vocals.wav", out / "no_vocals.wav"
    if not (vocals.exists() and inst.exists()):
        run([DEMUCS_PY, "-m", "demucs.separate", "-n", SEPARATION_MODEL, "--two-stems", "vocals",
             "--int24", "-o", SEP_DIR, track])
    return vocals, inst

def convert_voice(src_wav, dst_wav):
    """Turn the vocal in src_wav into your voice with Applio/RVC."""
    cmd = [sys.executable, "core.py", "infer", "--input-path", src_wav, "--output-path", dst_wav,
           "--pth-path", PTH, "--index-path", INDEX, "--pitch", PITCH_SHIFT, "--index-rate", INDEX_RATE,
           "--protect", PROTECT, "--f0-method", F0_METHOD, "--export-format", "WAV"]
    if SPLIT_AUDIO:
        cmd.append("--split-audio")
    run(cmd, cwd=APPLIO_DIR)

# ---- find the jobs: list of (name, vocal_source, instrumental_source or None)
jobs = []
if MODE == "full_song":
    for t in list_audio(FOLDERS["tracks"]):
        jobs.append((t.stem, t, None))
else:
    insts = {p.stem: p for p in list_audio(FOLDERS["instrumentals"])}
    for v in list_audio(FOLDERS["vocals"]):
        if v.stem in insts:
            jobs.append((v.stem, v, insts[v.stem]))
        else:
            print(f"⚠️  vocals/{v.name} has no instrumental with the same name - skipped")
if not jobs:
    raise SystemExit("Nothing to process - put songs in MyVoice/tracks (full_song) or matching files in MyVoice/vocals + MyVoice/instrumentals.")
print(f"{len(jobs)} song(s) to process\n")

results, failures = [], []
for name, vocal_src, inst_src in jobs:
    print(f"================ {name} ================")
    try:
        job_dir = WORK / "jobs" / re.sub(r"[^A-Za-z0-9_-]+", "_", name)
        job_dir.mkdir(parents=True, exist_ok=True)

        if MODE == "full_song":
            print("1/3 Separating vocals from the instrumental...")
            vocal_wav, inst_wav = separate(vocal_src)
        else:
            vocal_wav, inst_wav = job_dir / "vocal_in.wav", job_dir / "inst_in.wav"
            to_wav(vocal_src, vocal_wav)
            to_wav(inst_src, inst_wav)

        if PREVIEW_SECONDS > 0:
            print(f"   Preview mode: {PREVIEW_SECONDS}s starting at {PREVIEW_START}s")
            cut = []
            for src_wav, label in ((vocal_wav, "vocal_cut.wav"), (inst_wav, "inst_cut.wav")):
                dst_wav = job_dir / label
                subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(PREVIEW_START), "-t", str(PREVIEW_SECONDS),
                                "-i", str(src_wav), "-c:a", "pcm_f32le", str(dst_wav)], check=True)
                cut.append(dst_wav)
            vocal_wav, inst_wav = cut

        if CONVERT_VOCALS:
            print("2/3 Converting the vocal to your voice...")
            converted = job_dir / "vocal_converted.wav"
            convert_voice(vocal_wav, converted)
        else:
            converted = vocal_wav

        print("3/3 Mixing...")
        mix, vocal_fx, inst = mix_vocal_over_instrumental(
            converted, inst_wav, vocal_db=VOCAL_DB, inst_db=INSTRUMENTAL_DB, offset_ms=VOCAL_OFFSET_MS,
            reverb=REVERB, compress=COMPRESS_VOCAL, target_lufs=TARGET_LUFS)

        out_dir = FOLDERS["output"] / name
        out_dir.mkdir(parents=True, exist_ok=True)
        if PREVIEW_SECONDS > 0:
            final = save_audio(mix, out_dir / f"{name} ({MODEL_NAME}) PREVIEW pitch{PITCH_SHIFT:+d}", OUTPUT_FORMAT)
        else:
            final = save_audio(mix, out_dir / f"{name} ({MODEL_NAME})", OUTPUT_FORMAT)
            save_audio(vocal_fx, out_dir / f"{name} - my vocal only", "wav")
            save_audio(inst, out_dir / f"{name} - instrumental", "wav")
        results.append(final)
        print(f"✅ {final}\n")
    except Exception as e:
        failures.append((name, e))
        print(f"❌ {name} failed: {e}\n")

print("=" * 60)
print(f"Done: {len(results)} succeeded, {len(failures)} failed. Results are in {FOLDERS['output']}")
for f in results:
    print("\n" + f.name)
    display(Audio(str(f)))
for name, e in failures:
    print(f"FAILED {name}: {e}")

### Tuning guide
- **Start with the preview** (`PREVIEW_SECONDS` = 20). Each preview is saved as `... PREVIEW pitch+N`, so you can compare shifts like -12, -5, 0, +5, +12.
- Backing vocals and harmonies are part of the vocal stem, so they are converted to your voice too.
- **Wrong octave / strained high notes** → change `PITCH_SHIFT` (±12 moves a full octave).
- **Still sounds like the original singer** → raise `INDEX_RATE` (0.6-0.8).
- **Metallic / robotic / "s" sounds harsh** → lower `INDEX_RATE`, raise `PROTECT` towards 0.5, or use an earlier epoch (notebook 2 → `EXPORT_EPOCH`).
- **Leftover instrument bleed in the vocal** → try `SEPARATION_MODEL = htdemucs_ft`.
- **Vocal sits too loud / quiet** → `VOCAL_DB`. **Too dry / too wet** → `REVERB`.
- Output files are on Drive: `MyVoice/output`.